# Improve the Classifier

In the previous notebook, we trained an XGBoost classifier on the Amazon on-time delivery dataset and evaluated it with accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix. The baseline model reached roughly **83% accuracy** and **0.91 ROC-AUC**. We also saw that a one-feature rule on `total_items` reaches about 81% accuracy, because the label counts late items per truck.

This page tries the techniques most commonly used in industry to push those numbers higher, and measures each one on the same test set. It ends with a table that compares all of them against the baseline. Just as importantly, it covers how to make the model more robust and useful for the business.

:::{important} What "better" means
Before optimizing, decide which metric matters most for the **business decision** the model supports. For on-time prediction, recall on the "Delayed" class might be more valuable than overall accuracy if the cost of a missed delay (an unhappy customer) is higher than the cost of a false alarm (an unnecessary intervention).
:::

## A Map of the Techniques

The right-hand column describes what each technique *usually* does on tabular problems. Whether it helps on this dataset is an empirical question, which the rest of the page answers.

```{list-table}
:header-rows: 1

* - Approach
  - When to try it
  - Typical effect (in general)
* - Hyperparameter tuning
  - You have a working model and want to squeeze out more performance
  - Modest, broad improvements to accuracy / AUC / F1
* - Feature engineering
  - You suspect important business signal is hidden in the raw columns
  - Often the largest single gain
* - Handling class imbalance
  - One class is much rarer than the other
  - Improves recall on the minority class
* - Threshold tuning
  - Cost of false positives ≠ cost of false negatives
  - Trades precision against recall without retraining
* - Cross-validation
  - You want a more reliable estimate of generalization
  - Reduces variance of reported metrics
* - Regularization & early stopping
  - The model overfits the training data
  - Smaller train–test gap, better held-out AUC
* - Ensembling / model stacking
  - You have several decent models with different strengths
  - Small but consistent gains in F1 / AUC
```

## 1. Hyperparameter Tuning

XGBoost exposes many hyperparameters, and the defaults are rarely optimal for a specific dataset. The hyperparameters with the largest practical impact are usually:

- `n_estimators`: number of boosting rounds
- `learning_rate`: shrinkage applied to each new tree
- `max_depth`: depth of each tree
- `min_child_weight`: minimum sum of instance weight in a leaf
- `subsample`: fraction of rows sampled per tree
- `colsample_bytree`: fraction of columns sampled per tree
- `reg_alpha` (L1) and `reg_lambda` (L2): regularization

There are three search strategies you will see in practice:

1. **Grid search**: exhaustive over a small, hand-picked grid. Simple, but expensive.
2. **Random search**: sample combinations randomly. Usually finds a near-optimal point much faster than grid search.
3. **Bayesian / sequential search** (e.g., Optuna, Hyperopt): uses past trials to choose the next combination intelligently. State of the art for tabular models.

:::{tip} Pair tuning with cross-validation
Tune hyperparameters using cross-validated scores rather than a single train/test split. Otherwise you risk picking parameters that overfit the validation fold.
:::


## Setting Up

We start by reproducing the train/test split and the features from the [previous notebook](./build-a-classifier.ipynb). We also split a **validation set** off the training set. Early stopping and threshold tuning both need data that the model was not trained on, and the test set must stay untouched until the final comparison.

In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Load the Amazon on-time delivery dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/main/amazon-on-time-delivery-data/on-time-delivery-data.csv"
)

# Same features as the previous notebook: drop the constant precipitation_rate
# and replace zipcode with its first digit (a broad US region)
X = df.drop(columns=["classification_ontime", "precipitation_rate", "zipcode"])
X["zip_region"] = (df["zipcode"] // 10000).astype("category")

# Encode the target and create the same stratified split
le = LabelEncoder()
y = le.fit_transform(df["classification_ontime"])
print("Class encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Split the training set again: X_tr to fit models, X_val to validate them
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Fit part / validation part of train: {X_tr.shape} / {X_val.shape}")

Class encoding: {'Delayed': np.int64(0), 'On time': np.int64(1)}
Train: (36000, 14), Test: (9000, 14)
Fit part / validation part of train: (28800, 14) / (7200, 14)


We fit the same baseline XGBoost model as in the previous notebook. Every technique below is scored by the same `evaluate` function on the same test set, and the results are collected in a list called `results` for the comparison table at the end.

Besides accuracy and ROC AUC, we report F1 for "On time" (the positive class) and recall for "Delayed", because the business case above cares most about catching delays.

In [2]:
results = []


def evaluate(name, model, X_eval=X_test, threshold=0.5):
    """Score a fitted model on the test set and store the result."""
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)
    results.append(
        {
            "technique": name,
            "accuracy": accuracy_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "f1 (On time)": f1_score(y_test, pred),
            "recall (Delayed)": recall_score(y_test, pred, pos_label=0),
            "threshold": threshold,
        }
    )
    return pd.DataFrame(results[-1:]).set_index("technique").round(3)


baseline_params = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    enable_categorical=True,
    random_state=42,
)

baseline = XGBClassifier(**baseline_params)
baseline.fit(X_train, y_train)
evaluate("Baseline (previous notebook)", baseline)

,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
Baseline (previous notebook),0.828,0.908,0.799,0.838,0.5


### Random Search

We try 10 random combinations from the grid below, each scored by 3-fold cross-validation on the training set. That is 30 fits, plus one final refit of the best combination on the whole training set. A real search would try more combinations (50 to 100 is common), but 10 keeps the run time to a few minutes. The test set is used once, to score the refitted best model.

In [3]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [200, 400, 600, 800],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        enable_categorical=True,
        random_state=42,
    ),
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    n_jobs=1,  # XGBoost already uses all CPU cores; avoid nested parallelism
    random_state=42,
)

search.fit(X_train, y_train)
print("Best cross-validated ROC-AUC:", round(search.best_score_, 4))
print("Best params:", search.best_params_)

evaluate("Random search (10 candidates)", search.best_estimator_)

Best cross-validated ROC-AUC: 0.9096
Best params: {'subsample': 0.8, 'reg_lambda': 5.0, 'n_estimators': 400, 'min_child_weight': 10, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.8}


,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
Random search (10 candidates),0.831,0.91,0.802,0.841,0.5


## 2. Early Stopping

Rather than fixing `n_estimators` ahead of time, you can let XGBoost stop adding trees when validation performance plateaus. This usually gives a better-regularized model and saves training time.

The validation rows must be rows the model is **not** trained on. We fit on `X_tr` and let XGBoost watch ROC AUC on `X_val`. If you pass training rows as the evaluation set, the score keeps improving as the model memorizes them, and early stopping never does its job.

In [4]:
early_stop = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    early_stopping_rounds=50,
    eval_metric="auc",
    enable_categorical=True,
    random_state=42,
)

early_stop.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print("Best iteration:", early_stop.best_iteration)
print("Validation ROC-AUC at the best iteration:", round(early_stop.best_score, 4))

# predict_proba uses the trees up to the best iteration
evaluate("Early stopping (fit on X_tr)", early_stop)

Best iteration: 579
Validation ROC-AUC at the best iteration: 0.901


,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
Early stopping (fit on X_tr),0.83,0.908,0.802,0.839,0.5


## 3. Feature Engineering

Feature engineering is often the highest-leverage thing you can do for a tabular model. A few patterns that work well on operational data like this:

- **Differences and interactions**: combine related columns, e.g., the daily temperature range (`max_temperature - min_temperature`) or the gap between the temperature and the dew point (a rough fog indicator).
- **Weather flags and bins**: convert continuous measurements into business-meaningful groups, e.g., a "below freezing" flag or freezing / cold / mild / hot bins.
- **Location features**: we already use the first digit of the ZIP code. The first three digits give a finer region.
- **Target encoding** for high-cardinality categoricals such as `zipcode`. Use cross-fold target encoding to avoid leakage. Here most ZIP codes appear only about three times, so the encoded values would be noisy.
- **Log or square-root transforms** of skewed features help linear models. They do not change a tree model's splits, so they rarely help XGBoost.

Some common ideas are not possible with this file. There is no date column, so there are no calendar or seasonal features. There is no truck or route ID, so there is no per-truck history. And `precipitation_rate` is always zero, so it cannot be combined with anything.

:::{caution} Avoid leakage
Any feature you build must be computable using **only information available at prediction time**. For example, an "average delay rate per zipcode" must be calculated only from training data, never from the test set.
:::

Below we add four simple weather features and retrain the baseline settings.

In [5]:
def add_weather_features(frame):
    out = frame.copy()
    out["temp_range"] = out["max_temperature"] - out["min_temperature"]
    out["dew_point_spread"] = out["temperature"] - out["dew_point_temperature"]
    out["below_freezing"] = (out["min_temperature"] <= 0).astype(int)
    out["has_snow"] = (out["snow_depth"] > 0).astype(int)
    return out


X_train_fe = add_weather_features(X_train)
X_test_fe = add_weather_features(X_test)

fe_model = XGBClassifier(**baseline_params)
fe_model.fit(X_train_fe, y_train)
evaluate("Feature engineering (4 weather features)", fe_model, X_eval=X_test_fe)

,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
Feature engineering (4 weather features),0.828,0.907,0.799,0.837,0.5


## 4. Handling Class Imbalance

Our target is moderately imbalanced (~58% Delayed / ~42% On time). For more skewed problems, three techniques are common:

1. **`scale_pos_weight`** in XGBoost: multiplies the weight of the positive class. The usual starting value, `count(negatives) / count(positives)`, gives both classes the same total weight.
2. **Resampling**: oversample the minority class (e.g., `imblearn.SMOTE`) or undersample the majority class.
3. **Cost-sensitive learning**: pass `sample_weight` to `.fit()` so that misclassifying the rare class incurs a larger penalty.

:::{caution} Check which class you are up-weighting

In our encoding the positive class is "On time" (1), which is also the minority class. Balancing with `scale_pos_weight = count(Delayed) / count(On time)` therefore gives *more* weight to on-time deliveries. Expect higher recall on "On time" and lower recall on "Delayed", which is the opposite of what the business case at the top of this page asks for. If catching delays matters most, up-weight "Delayed" instead (a `scale_pos_weight` below 1), or move the decision threshold, as in the next section.

:::

In [6]:
n_delayed = (y_train == 0).sum()
n_on_time = (y_train == 1).sum()
spw = n_delayed / n_on_time
print(f"scale_pos_weight = {spw:.2f}")

weighted = XGBClassifier(**baseline_params, scale_pos_weight=spw)
weighted.fit(X_train, y_train)
evaluate("Class weights (scale_pos_weight)", weighted)

scale_pos_weight = 1.39


,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
Class weights (scale_pos_weight),0.824,0.907,0.804,0.797,0.5


## 5. Threshold Tuning

`predict()` uses a default decision threshold of 0.5. Moving the threshold lets you trade precision for recall **without retraining**.

The threshold is a setting of the model, so it must be chosen on validation data, never on the test set. If you pick the threshold that maximizes F1 on the test set and then report that F1, the score is optimistic: the threshold was fitted to those very rows.

We fit the baseline settings on `X_tr`, choose the threshold that maximizes F1 for "On time" on `X_val`, and apply that fixed threshold to the test set once. Because this model is trained on fewer rows than the baseline, we also score it at the default threshold of 0.5, so the effect of the threshold alone is visible.

In [7]:
from sklearn.metrics import precision_recall_curve

tuned = XGBClassifier(**baseline_params)
tuned.fit(X_tr, y_tr)

# Choose the threshold on the validation set
val_proba = tuned.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, val_proba)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = np.argmax(f1[:-1])  # the last precision/recall pair has no threshold
best_threshold = float(thr[best_idx])
print(f"Threshold chosen on validation: {best_threshold:.3f}")
print(f"Validation F1 (On time) at that threshold: {f1[best_idx]:.3f}")

# Score the same model on the test set at 0.5 and at the chosen threshold
evaluate("Fit on X_tr, threshold 0.5", tuned)
evaluate("Fit on X_tr, threshold tuned on validation", tuned, threshold=best_threshold)

Threshold chosen on validation: 0.348
Validation F1 (On time) at that threshold: 0.798


,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
"Fit on X_tr, threshold tuned on validation",0.814,0.907,0.801,0.756,0.348


:::{tip} Pick a threshold that reflects business cost

Maximizing F1 is a convenient default, not a business objective. If a missed delay costs the company 10× more than a false alarm, choose the threshold on the validation set that minimizes `10 * FN + 1 * FP` (with "Delayed" as the class you want to catch) rather than the one that maximizes F1.

:::

## 6. Cross-Validation

A single 80/20 split gives one estimate of model quality. **Stratified k-fold cross-validation** gives `k` estimates and reports their mean and spread, which is more reliable. We run it on the training set only, so the test set stays untouched.

The spread is also a yardstick for the comparison table at the end: a difference between two techniques that is much smaller than the fold-to-fold spread is probably noise.

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_validate(
    XGBClassifier(**baseline_params),
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "roc_auc"],
)
acc, auc = cv_scores["test_accuracy"], cv_scores["test_roc_auc"]
print(f"Accuracy: {acc.mean():.3f} ± {acc.std():.3f}")
print(f"ROC-AUC:  {auc.mean():.3f} ± {auc.std():.3f}")

Accuracy: 0.832 ± 0.004
ROC-AUC:  0.910 ± 0.002


## 7. Try a Different Model Family

XGBoost is an excellent default for tabular data, but it is worth comparing against the following (we do not run them here):

- **LightGBM**: often faster than XGBoost, with comparable accuracy.
- **CatBoost**: handles categorical features natively, useful when you have many high-cardinality columns like `zipcode`.
- **Logistic regression** with engineered features: a strong, interpretable baseline.
- **Random forest**: less prone to overfitting than a single tree, useful as a sanity check.

A quick comparison table on the same train/test split is often the fastest way to see whether you are on the right track.

## 8. Ensembling

Once you have several reasonable models, you can combine them:

- **Soft voting**: average the predicted probabilities of multiple models.
- **Stacking**: train a "meta-model" (often logistic regression) on the out-of-fold predictions of base learners.

Ensembles rarely produce dramatic gains. They can add a little on top of the best single model and reduce variance, at the cost of a more complex system to run.

## A Suggested Workflow

For most tabular projects, a productive order of operations is:

1. Lock in a clean train/validation/test split (or cross-validation scheme).
2. Get a simple model (logistic regression or default XGBoost) working end to end.
3. Invest in **feature engineering** until improvements taper off.
4. Address **class imbalance** if relevant.
5. Run **hyperparameter search** with **early stopping** and cross-validation.
6. **Tune the decision threshold** to match the business cost structure.
7. Compare against alternative model families and consider **ensembling** the best two or three.

## How Classifiers Are Used in the Real World

Binary and multi-class classifiers are among the most widely deployed machine-learning systems at large companies. A few well-documented examples:

:::{seealso} Real-world classifier examples

**Amazon: Fulfillment, fraud, and recommendations**
: Amazon uses classifiers throughout its operations. Within fulfillment, models predict whether a shipment will arrive on time so the company can proactively reroute orders or notify customers. Amazon's [Fraud Detector](https://aws.amazon.com/fraud-detector/) service productizes the same underlying technology: gradient-boosted classifiers that score events as fraudulent or legitimate in real time.

**Netflix: Personalization**
: Netflix uses classifiers to decide which artwork to show for a given title, which row of recommendations to surface, and whether a user is likely to abandon a session. The company has [written extensively](https://netflixtechblog.com/) about how small classification gains translate into large engagement and retention improvements.

**Google: Spam and abuse detection**
: Gmail uses deep neural-network classifiers to filter spam and phishing, [blocking more than 99.9% of spam, phishing, and malware](https://workspace.google.com/blog/identity-and-security/improving-spam-filter-with-tensorflow). Similar models power abuse detection across YouTube and Google Ads.

**Meta: Content moderation and ad ranking**
: Meta runs classifiers that flag potentially policy-violating content (hate speech, graphic violence, misinformation) at the scale of billions of posts per day. Separate classifier stacks predict the probability that a user will click, like, or comment on a piece of content, which directly drives ad and feed ranking.

**Uber: ETA confidence and fraud**
: Uber uses gradient-boosted classifiers to predict whether a trip's actual arrival time will deviate significantly from the estimate, and to flag suspicious driver or rider accounts. The [Michelangelo platform](https://www.uber.com/blog/michelangelo-machine-learning-platform/) provides shared infrastructure for training and serving these models.

**Airbnb: Trust and search ranking**
: Airbnb uses classifiers to estimate the probability that a booking is fraudulent, that a host will accept a request, and that a guest will book a given listing. These signals feed both the trust system and the [search ranking model](https://medium.com/airbnb-engineering/).

**Capital One and JPMorgan Chase: Credit and fraud**
: Major US banks use classifiers to decide whether to approve a credit application and to score every card transaction in real time for fraud. Modern stacks combine gradient-boosted trees with deep learning, evaluated primarily on precision–recall metrics rather than accuracy because fraud is highly imbalanced.

**UPS and FedEx: Logistics**
: Shipping companies use on-time-delivery classifiers very similar to the model in this case study to forecast service exceptions, reroute parcels, and staff distribution centers ahead of weather events.

:::

The common thread across these examples is that the **business decision drives the metric**. None of these companies optimize raw accuracy; they optimize a composite that reflects the dollar cost of false positives versus false negatives, the latency budget of the serving system, and how the model's output will be acted on. As you improve the Amazon delivery classifier, frame each change the same way: which metric does it move, and why does that matter for the customer or the business?


## Results: Did Any Technique Help?

The table collects every model scored above. All rows use the same 9,000-row test set. The threshold is 0.5 except in the row where it was chosen on the validation set.

In [9]:
df_results = pd.DataFrame(results).set_index("technique").round(3)
df_results

,accuracy,roc_auc,f1 (On time),recall (Delayed),threshold
technique,,,,,
Baseline (previous notebook),0.828,0.908,0.799,0.838,0.500
Random search (10 candidates),0.831,0.910,0.802,0.841,0.500
Early stopping (fit on X_tr),0.830,0.908,0.802,0.839,0.500
Feature engineering (4 weather features),0.828,0.907,0.799,0.837,0.500
Class weights (scale_pos_weight),0.824,0.907,0.804,0.797,0.500
"Fit on X_tr, threshold 0.5",0.828,0.907,0.799,0.837,0.500
"Fit on X_tr, threshold tuned on validation",0.814,0.907,0.801,0.756,0.348


## Conclusion

**None of the techniques improved the model in a meaningful way.**

- **Accuracy and ROC AUC barely move.** Random search and early stopping add a few tenths of a percentage point of accuracy at most. That is about the size of the fold-to-fold spread in the cross-validation above, so we cannot tell these gains apart from noise. The four extra weather features change nothing.
- **Class weights and threshold tuning trade one class against the other.** Both push the model toward predicting "On time" more often. F1 for "On time" stays about the same, while recall on "Delayed" falls by several points. For a business that wants to catch delays, that is a step in the wrong direction. Choosing the threshold with a business cost that penalizes missed delays (see the tip in section 5) would move recall on "Delayed" up instead.
- **The 80% accuracy goal was already met by the baseline**, and, as the previous notebook showed, almost by a one-feature rule on `total_items`.

The ceiling here is set by the data, not by the model. The label counts late items per truck, so `total_items` carries most of the signal, and the weather columns add little. Getting a materially better model would take new information, such as dates, truck or route IDs, or a label that does not grow with the number of items on the truck, rather than more tuning.

This is a common outcome in practice. Measuring each technique against a baseline on the same test set is what lets you see it, and stop spending effort on tuning that does not pay off.